# 02 — Preprocessing & Imbalance Strategies

This notebook builds the shared preprocessing pipeline (scaling, 3-way split) and compares three approaches to handling class imbalance: SMOTE, class weighting, and threshold tuning.

## 1. Load and Clean Data

Reload the raw dataset and reapply the two fixes established in EDA: casting `Class` to integer and removing the 1,081 duplicate rows. This notebook is independent of `01_eda.ipynb`, so these steps are repeated here — once we move this logic into `src/features.py`, later notebooks will just import it instead of repeating it.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/raw/creditcard.csv')
df['Class'] = df['Class'].astype(str).str.strip("'").astype(int)
df = df.drop_duplicates()

print(df.shape)
print(df['Class'].value_counts())

(283726, 31)
Class
0    283253
1       473
Name: count, dtype: int64


## 2. Scale `Amount` and `Time`

The `V1`–`V28` columns are already PCA components (roughly zero-mean, unit-variance) and should **not** be scaled again. Only `Amount` and `Time` are raw, unscaled features — apply `StandardScaler` to just these two so they're on a comparable scale to the PCA features for models that are scale-sensitive (Logistic Regression, and later the conformal prediction calibration).

In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[['Amount', 'Time']] = scaler.fit_transform(df[['Amount', 'Time']])

df[['Amount', 'Time']].describe()

,Amount,Time
count,2.837260e+05,2.837260e+05
mean,-8.013847e-19,1.538659e-16
std,1.000002e+00,1.000002e+00
min,-3.533268e-01,-1.996823e+00
25%,-3.309625e-01,-8.552128e-01
50%,-2.654671e-01,-2.131081e-01
75%,-4.378088e-02,9.369423e-01
max,1.022476e+02,1.642362e+00


## 3. Three-Way Stratified Split: Train / Calibration / Test

Split the data into three sets instead of the usual two. The calibration set is required later for the conformal prediction wrapper (Layer 4) — MAPIE needs data the model hasn't trained on to produce valid confidence intervals.

Split ratios: 60% train / 20% calibration / 20% test. `stratify=` is essential here — with only 473 fraud cases in the entire dataset, a non-stratified split could easily leave one of the three sets with very few (or disproportionate) fraud examples.

In [3]:
X = df.drop(columns=['Class'])
y = df['Class']

# First split: 60% train, 40% temp (which becomes calibration + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)

# Second split: split the 40% temp into 20% calibration, 20% test
X_calib, X_test, y_calib, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("Train:", X_train.shape, y_train.sum(), "fraud")
print("Calibration:", X_calib.shape, y_calib.sum(), "fraud")
print("Test:", X_test.shape, y_test.sum(), "fraud")

Train: (170235, 30) 284 fraud
Calibration: (56745, 30) 94 fraud
Test: (56746, 30) 95 fraud
